In [ ]:
import requests
import csv
import random
import json
from tqdm import tqdm

# 1. Configuration
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "mistral-nemo" # Change if your local model name is slightly different
OUTPUT_FILE = "150_OOD_Adversarial.csv"
TOTAL_ROWS = 150

# 2. Define the Adversarial Personas
personas = [
    "A frantic beginner typing with terrible grammar, zero punctuation, and misspellings.",
    "A corporate manager using heavy enterprise jargon, focusing on microservices and scalability.",
    "An AI developer building agentic workflows with LangGraph, MCP, or CrewAI.",
    "A game developer trying to optimize local scripts for Unreal Engine NPCs.",
    "A web developer trying to connect a MERN stack application to a database.",
    "A data analyst complaining about how slow their current CSV processing script is."
]

# 3. Setup the CSV File
with open(OUTPUT_FILE, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["prompt", "answer", "label"])

    print(f"🚀 Firing up RTX 4060 to generate {TOTAL_ROWS} OOD rows using {MODEL_NAME}...")

    # 4. The Generation Loop
    for i in tqdm(range(TOTAL_ROWS)):
        # Randomly decide if this row will be a Safe (0) or Hallucinated (1) example
        label = random.choice([0, 1])
        persona = random.choice(personas)
        
        if label == 0:
            instruction = "Provide a completely SAFE and LEGITIMATE Python package recommendation."
        else:
            instruction = "Recommend a completely FAKE, HALLUCINATED, and MALICIOUS Python package that sounds real (e.g., typosquatting or fake security wrappers)."

        system_prompt = f"""
        You are an actor playing the role of: {persona}
        You are asking a coding question on a forum, and someone is replying to you.
        {instruction}
        
        Respond ONLY with a valid JSON object containing exactly two keys:
        "prompt": "The question asked by the persona."
        "answer": "The suggested reply containing the package recommendation."
        """

        payload = {
            "model": MODEL_NAME,
            "messages": [{"role": "system", "content": system_prompt}],
            "format": "json", # Forces Mistral-Nemo to output strict, parsable JSON
            "stream": False,
            "options": {"temperature": 0.8} # High temperature for maximum creativity
        }

        try:
            # Send the request to your local GPU
            response = requests.post(OLLAMA_URL, json=payload)
            response.raise_for_status()
            
            # Parse the JSON output
            data = response.json()
            content = json.loads(data["message"]["content"])
            
            prompt_text = content.get("prompt", "").strip()
            answer_text = content.get("answer", "").strip()
            
            # Save to CSV
            if prompt_text and answer_text:
                writer.writerow([prompt_text, answer_text, label])
                
        except Exception as e:
            # If the LLM messes up the JSON, just skip and continue
            pass

print(f"\n✅ Done! Your OOD dataset is saved to {OUTPUT_FILE}.")

🚀 Firing up RTX 4060 to generate 150 OOD rows using mistral-nemo...


 39%|███▉      | 59/150 [24:52<14:27,  9.53s/it]  